In [1]:
import pickle

# Load embedding
# review collaborative embeddings: "../../data/games_with_embeddings.pkl"
# game description embeddings: "../../data/game_descriptions_embeddings.pkl"
# game title embeddings: "../../data/game_title_embeddings.pkl"
with open("../../data/game_title_embeddings.pkl", "rb") as f:
    game_embedding_dict = pickle.load(f)

In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_closest_games(target_game, embeddings_dict, top_n=5):
    """
    Finds the most similar games based on their embedding vectors.
    
    :param target_game: Name of the game to search for (str)
    :param embeddings_dict: Dictionary mapping Game Name -> Numpy Array (size 32)
    :param top_n: Number of recommendations to return
    """
    
    # Check if the game exists in our database
    if target_game not in embeddings_dict:
        return f"Error: '{target_game}' not found in the embeddings database."

    # Isolate the target vector and reshape it for sklearn
    # reshape(1, -1) turns it from shape (32,) to (1, 32)
    target_vector = embeddings_dict[target_game].reshape(1, -1)
    
    # Prepare the rest of the data
    # We separate names and vectors so their indexes match up
    game_names = list(embeddings_dict.keys())
    all_vectors = np.array(list(embeddings_dict.values()))
    
    # Calculate Cosine Similarity
    # This compares the target (1, 32) against all games (N, 32) simultaneously
    # It returns an array of scores from -1.0 (opposites) to 1.0 (identical)
    similarity_scores = cosine_similarity(target_vector, all_vectors)[0]
    
    # Sort the results
    # argsort() gives us the indexes from lowest to highest, so we reverse it [::-1]
    ranked_indexes = np.argsort(similarity_scores)[::-1]
    
    # Format the output
    print(f"Games most similar to '{target_game}':\n")
    results = []
    
    for idx in ranked_indexes:
        match_name = game_names[idx]
        score = similarity_scores[idx]
        
        # Skip the target game itself (it will always have a 1.0 score)
        if match_name == target_game:
            continue
            
        results.append((match_name, score))
        print(f"{len(results)}. {match_name} (Similarity Score: {score:.4f})")
        
        # Stop once we hit our desired number of recommendations
        if len(results) == top_n:
            break
            
    return results

In [10]:
# For reproducible random numbers
np.random.seed(42) 

# Run the function
find_closest_games("Horizon Zero Dawn: The Frozen Wilds", game_embedding_dict, top_n=10)

Games most similar to 'Horizon Zero Dawn: The Frozen Wilds':

1. Horizon Zero Dawn (Similarity Score: 0.6763)
2. Crysis 3: The Lost Island (Similarity Score: 0.6545)
3. Lost Horizon 2 (Similarity Score: 0.6266)
4. Forza Horizon 3: Blizzard Mountain (Similarity Score: 0.6256)
5. Horizon Zero Dawn Remastered (Similarity Score: 0.6243)
6. Monster Hunter World: Iceborne (Similarity Score: 0.6143)
7. Glacier 3: The Meltdown (Similarity Score: 0.6130)
8. Left 4 Dead 2: Cold Stream (Similarity Score: 0.6126)
9. Horizon Zero Dawn Complete Edition (Similarity Score: 0.6083)
10. FrostBound (Similarity Score: 0.6042)


[('Horizon Zero Dawn', np.float32(0.6762815)),
 ('Crysis 3: The Lost Island', np.float32(0.6544681)),
 ('Lost Horizon 2', np.float32(0.62655926)),
 ('Forza Horizon 3: Blizzard Mountain', np.float32(0.6255529)),
 ('Horizon Zero Dawn Remastered', np.float32(0.6243001)),
 ('Monster Hunter World: Iceborne', np.float32(0.61430746)),
 ('Glacier 3: The Meltdown', np.float32(0.61302173)),
 ('Left 4 Dead 2: Cold Stream', np.float32(0.6126267)),
 ('Horizon Zero Dawn Complete Edition', np.float32(0.6083422)),
 ('FrostBound', np.float32(0.60423636))]